For faster downloads and for better GPU, use colab extension on VS Code to connect to a remote Colab Instance

# Checking the Non domain-adapted model

ModernBERT-base is used for its benchmarks and size.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, default_data_collator, DataCollatorForLanguageModeling
from datasets import Dataset
import collections
import numpy as np
import math

SEED = 67
CHUNK_SIZE = 128
WWM_PROB = 0.2
BATCH_SIZE = 64 # GPU bound

In [2]:

model_use = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_use)
model = AutoModelForMaskedLM.from_pretrained(model_use)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Attempt on an arbitrary text on a news body

In [3]:
test = " Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week [MASK] as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte\u2019s desk"

In [4]:
inputs = tokenizer(test, return_tensors='pt')
outputs = model(**inputs)
logits = outputs.logits

In [ ]:
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer.decode([token])}, {test.replace(tokenizer.mask_token, tokenizer.decode(token).strip())}")

 stint,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week stint as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 tenure,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week tenure as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 term,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week term as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 assignment,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week assignment as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodri

## Preprocessing the Dataset

In [6]:
import jsonl
import pandas as pd



titles = []
articles = []

sources = ["abscbn", "gma", "rappler", "upparser"]
for source in sources:
    with open(f'./parawrite/scraping/{source}_articles.jsonl', encoding='utf-8', errors="replace") as source_f:
        source_json = jsonl.load(source_f)
        for article in source_json:
            t = article['title']
            b = article['body']
            # print(b)
            titles.append(t)
            articles.append(b)

articles_df = pd.DataFrame({
    "title": titles,
    "body": articles
})

articles_df

,title,body
0,Iran minister heads to Russia as talks remain ...,Iran's foreign minister headed to Russia on Su...
1,Dizon orders immediate declogging of SLEX drai...,Dizon said the DPWH will work together with SL...
2,Marcos to tackle China's gray zone tactics wit...,Marcos said he wants to find out more about 't...
3,11 of 19 Toboso fatalities tested positive for...,"Police Colonel Reynaldo Calaoa, chief of the P..."
4,"Leviste unhappy in Congress, open to other pos...",Speaking at the Kapihan ng Samahang Plaridel i...
...,...,...
34528,The UP Parser Receives Best Student Publicatio...,The UP Parser was awarded twice during the Eng...
34529,"Bootcamp 5.0: Freshies, Assemble!",Bootcamp 5.0 was held at the UP Department of ...
34530,CURSOR Week,Episode 2 of 10 (1516A): In this episode of CS...
34531,UP CURSOR Launches its Newest Logo,The UP Association of Computer Science Majors ...


In [7]:
def tokenize_func(data):
    tokenized = tokenizer(data["body"])
    if tokenizer.is_fast:
        tokenized["word_ids"] = [tokenized.word_ids(i) for i in range(len(tokenized["input_ids"]))]
    return tokenized


articles_ds = Dataset.from_pandas(articles_df)
articles_ds = articles_ds.train_test_split(seed=SEED)

tokenized_articles = articles_ds.map(
    tokenize_func, batched=True, remove_columns=['title', 'body']
)
tokenized_articles


Map:   0%|          | 0/25899 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (9596 > 8192). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/8634 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids'],
        num_rows: 25899
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids'],
        num_rows: 8634
    })
})

Concatenate all articles and split each article into texts of length chunk size.

In [8]:
def concatenate_chunk(data):
    concatenated = {key: sum(data[key], []) for key in data}
    total_length = len(concatenated[list(data)[0]])
    # remove last when smaller than chunk size
    total_length = (total_length // CHUNK_SIZE) * CHUNK_SIZE
    # split
    splitted = {
        key: [row[i:i + CHUNK_SIZE] for i in range(0, total_length, CHUNK_SIZE)]
        for key, row in concatenated.items()
    }

    splitted["labels"] = splitted["input_ids"].copy()

    return splitted

In [9]:
articles_chunked = tokenized_articles.map(concatenate_chunk, batched=True)
articles_chunked

Map:   0%|          | 0/25899 [00:00<?, ? examples/s]

Map:   0%|          | 0/8634 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 136604
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 46421
    })
})

In [10]:
tokenizer.decode(articles_chunked["train"][0]["input_ids"])

"[CLS]Ayon sa driver ng 12-wheeler truck, patungo siya sa Pasig mula Pampanga at may kargang buhangin nang makarinig siya ng kalabog. 'Naramdaman ko na lang po may kumalabog sa ilalim ng truck ko. Dito po siya sa pinakakanan ko, sa mismong island,' paliwanag ng driver. Agad niyang inihinto ang trak at doon na nakita ang nagulungang rider. Dead-on-the-spot ang biktima"

## Fine-tuning

In [11]:
# perform whole word masking, not based on tokens, rather on "whole words" based on the collated words gathered during preprocessing
# Adapted from Hugging Face Documentation https://huggingface.co/learn/llm-course/chapter7/3
def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # create a map for a word to list of indices in tokenizer
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)
        
        # mask

        mask = np.random.binomial(1, WWM_PROB, (len(mapping), ))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels
    
    return default_data_collator(features)

### Training

In [12]:
logging_steps = len(articles_chunked["train"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

training_args = TrainingArguments(
    output_dir=f"{model_use.split('/')[-1]}-finetuned-all",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_eval_batch_size=BATCH_SIZE,
    per_device_train_batch_size=BATCH_SIZE,
    fp16=True,
    logging_steps=logging_steps
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=articles_chunked["train"],
    eval_dataset=articles_chunked["test"],
    data_collator=data_collator,
)

In [14]:
eval_results = trainer.evaluate()
print(f"Perplexity before fine-tuning: {math.exp(eval_results['eval_loss'])}")

Perplexity before fine-tuning: 8.539410216555021


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,1.457739,0.002600
2,No log,1.400394,0.002600
3,No log,1.387698,0.002600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6405, training_loss=1.4415251756440282, metrics={'train_runtime': 6870.9014, 'train_samples_per_second': 59.645, 'train_steps_per_second': 0.932, 'total_flos': 3.492703213800653e+16, 'train_loss': 1.4415251756440282, 'epoch': 3.0})

In [16]:
eval_results = trainer.evaluate()
print(f"Perplexity after fine-tuning: {math.exp(eval_results['eval_loss'])}")

Perplexity after fine-tuning: 4.022210057930634


In [17]:
trainer.save_model() 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch


tokenizer_test = AutoTokenizer.from_pretrained('./ModernBERT-base-finetuned-all')
model_test = AutoModelForMaskedLM.from_pretrained('./ModernBERT-base-finetuned-all')


test = "CS Week 2025 [MASK] with a blast as students conquer the halls with glee"
test = " Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week [MASK] as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte\u2019s desk"
test_input = tokenizer_test(test, return_tensors="pt")

test_output = model_test(**test_input)
test_logits = test_output.logits


mask_token_index = torch.where(test_input["input_ids"] == tokenizer_test.mask_token_id)[1]
mask_token_logits = test_logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer_test.decode([token])}, {test.replace(tokenizer_test.mask_token, tokenizer_test.decode(token).strip())}")

top_10 = torch.argsort(mask_token_logits, dim = 1, descending=True)
print(top_10)

for t in top_10.tolist():
    print(t)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

 stint,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week stint as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 term,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week term as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 tenure,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week tenure as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 visit,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week visit as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
